
# المرحلة الأولى: تجهيز البيانات ومقارنة استراتيجيات التقسيم

## الهدف

تهدف هذه المرحلة إلى تجهيز مجموعات البيانات العربية والإنجليزية الخاصة بنظام حماية البيانات الشخصية (PDPL)، والتحقق من بنيتها وجودتها، ومقارنة طريقتَي تقسيم النصوص: التقسيم الثابت (`Fixed-size Chunking`) والتقسيم حسب المادة أو القسم (`Article/Section Chunking`).

## نطاق العمل

1. تحميل مجموعات البيانات والتحقق من الحقول الأساسية.
2. تطبيع النصوص العربية لأغراض البحث مع الاحتفاظ بالنص الأصلي.
3. تجهيز المقاطع والبيانات الوصفية.
4. مقارنة عدد المقاطع وأطوالها وتوفر المعرّفات وأرقام المواد.
5. فحص المقاطع القصيرة والتحقق من سلامة المعرّفات.
6. حفظ المخرجات والتحقق من إمكانية قراءتها مجددًا.

## مجموعات البيانات

تتضمن البيانات أربع مجموعات للمقاطع العربية والإنجليزية، بالإضافة إلى مجموعة أسئلة وإجابات مرجعية (`Gold QA`) لاستخدامها في مرحلة التقييم اللاحقة.

## حدود المرحلة

تركز هذه المرحلة على تجهيز البيانات وفحصها ومقارنة استراتيجيات التقسيم. لا تتضمن بناء نظام الاسترجاع أو توليد الإجابات أو تقييم أداء النظام النهائي.


In [1]:

from pathlib import Path

BASE = Path("/kaggle/input/datasets/khadijah2000")

FILES = {
    "ar_fixed": BASE / "pdpl-chunking-comparison-datasets/fixed_size_chunks.json",
    "ar_article": BASE / "pdpl-chunking-comparison-datasets/article_section_chunks.json",
    "en_fixed": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json",
    "en_article": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json",
    "gold_qa": BASE / "pdpl-bilingual-evaluation-dataset/gold_qa (2).json",
}

for name, path in FILES.items():
    print("✅" if path.is_file() else "❌", name, "—", path)


✅ ar_fixed — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/fixed_size_chunks.json
✅ ar_article — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets/article_section_chunks.json
✅ en_fixed — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json
✅ en_article — /kaggle/input/datasets/khadijah2000/pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json
✅ gold_qa — /kaggle/input/datasets/khadijah2000/pdpl-bilingual-evaluation-dataset/gold_qa (2).json


In [2]:

# المرحلة 1: تحميل البيانات وفحص بنيتها

import json
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/datasets/khadijah2000")

FILES = {
    "ar_fixed": BASE / "pdpl-chunking-comparison-datasets/fixed_size_chunks.json",
    "ar_article": BASE / "pdpl-chunking-comparison-datasets/article_section_chunks.json",
    "en_fixed": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_fixed_chunks_en.json",
    "en_article": BASE / "pdpl-chunking-comparison-datasets-en/pdpl_9_documents_artical_chunks_en.json",
    "gold_qa": BASE / "pdpl-bilingual-evaluation-dataset/gold_qa (2).json",
}

raw_data = {}

for name, path in FILES.items():
    with path.open("r", encoding="utf-8") as f:
        raw_data[name] = json.load(f)
    print(f"تم تحميل {name}")

def get_records(data):
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("records", "chunks", "data", "items"):
            if isinstance(data.get(key), list):
                return data[key]

        return [data]

    return []

# نبقي المجموعات منفصلة ولا ندمجها الآن
retrieval_data = {
    name: get_records(raw_data[name])
    for name in ("ar_fixed", "ar_article", "en_fixed", "en_article")
}

summary = []

for name, records in retrieval_data.items():
    first = records[0] if records else {}

    text_fields = [
        field
        for field in ("search_text", "clean_text", "text")
        if any(str(r.get(field, "")).strip() for r in records)
    ]

    doc_ids = {
        r.get("document_id") or r.get("parent_id")
        for r in records
        if r.get("document_id") or r.get("parent_id")
    }

    summary.append({
        "dataset": name,
        "chunks": len(records),
        "documents_with_ids": len(doc_ids),
        "text_fields_found": ", ".join(text_fields),
        "article_number_field": "article_number" in first,
        "covered_article_numbers_field": "covered_article_numbers" in first,
        "sample_fields": ", ".join(first.keys()) if first else "لا توجد سجلات",
    })

display(pd.DataFrame(summary))

gold_records = get_records(raw_data["gold_qa"])
print("عدد سجلات gold_qa:", len(gold_records))

if gold_records:
    print("حقول أول سؤال:", list(gold_records[0].keys()))
    print("عينة من السؤال الأول:")
    print(json.dumps(gold_records[0], ensure_ascii=False, indent=2)[:2000])


تم تحميل ar_fixed
تم تحميل ar_article
تم تحميل en_fixed
تم تحميل en_article
تم تحميل gold_qa


,dataset,chunks,documents_with_ids,text_fields_found,article_number_field,covered_article_numbers_field,sample_fields
0,ar_fixed,241,10,"search_text, clean_text, text",True,False,"id, document_id, language, article_number, art..."
1,ar_article,196,10,"search_text, clean_text, text",True,False,"id, document_id, language, article_number, art..."
2,en_fixed,181,10,"search_text, clean_text",True,True,"chunk_id, document_id, language, chunking, chu..."
3,en_article,252,9,"search_text, clean_text",True,False,"chunk_id, parent_id, document_id, language, un..."


عدد سجلات gold_qa: 90
حقول أول سؤال: ['question_id', 'language', 'question', 'reference_answer', 'gold_evidence_extract', 'source_workbook', 'source_document', 'source_id_original', 'gold_reference_original', 'gold_document_id', 'gold_article_number', 'gold_references', 'answerable', 'in_corpus', 'evaluation_status', 'eligible_for_article_retrieval_eval', 'reference_type', 'question_type', 'pdf_page', 'review_notes', 'question_v2', 'revision_action', 'revision_reason', 'evidence_matching_status', 'reference_review_status', 'review_priority', 'reviewer_notes', 'approved_for_final_eval', 'best_chunk_id', 'evidence_review_status', 'gold_reference_in_chunk_corpus', 'manual_evidence_approval']
عينة من السؤال الأول:
{
  "question_id": "AR001",
  "language": "ar",
  "question": "ما المقصود بالبيانات الشخصية في النظام؟",
  "reference_answer": "كل بيان يمكن أن يؤدي إلى معرفة الفرد أو يجعل التعرف عليه ممكنًا مباشرة أو بصورة غير مباشرة، بصرف النظر عن مصدره أو شكله.",
  "gold_evidence_extract": "ك

In [3]:
# تجهيز جداول المقاطع وأسئلة التقييم قبل فحص المراجع

# التأكد من أن مصادر البيانات موجودة
assert "retrieval_data" in globals(), (
    "retrieval_data غير موجود؛ شغّلي خلية تحميل البيانات أولاً."
)
assert "raw_data" in globals(), (
    "raw_data غير موجود؛ شغّلي خلية تحميل البيانات أولاً."
)

# تحويل مجموعات المقاطع الأربع إلى جدول موحّد
chunk_frames = []

for dataset_name, records in retrieval_data.items():
    if records:
        df = pd.DataFrame(records).copy()
        df["dataset"] = dataset_name
        chunk_frames.append(df)

chunks_df = pd.concat(chunk_frames, ignore_index=True)

# تحويل أسئلة التقييم الذهبية إلى جدول
gold_records = get_records(raw_data["gold_qa"])
gold_df = pd.DataFrame(gold_records)

print("عدد المقاطع:", len(chunks_df))
print("عدد أسئلة التقييم:", len(gold_df))

print("\nأعمدة المقاطع:")
print(chunks_df.columns.tolist())

print("\nأعمدة أسئلة التقييم:")
print(gold_df.columns.tolist())

عدد المقاطع: 870
عدد أسئلة التقييم: 90

أعمدة المقاطع:
['id', 'document_id', 'language', 'article_number', 'article_title', 'text', 'search_text', 'source_url', 'html_sha256', 'extraction_status', 'clean_text', 'source_record_id', 'chunk_id', 'chunk_index', 'chunk_count', 'chunking_strategy', 'token_count', 'quality_flags', 'source_input_file', 'processed_at_utc', 'text_sha256', 'document_title', 'source_type', 'section_id', 'section_order', 'heading_level', 'content_selector', 'extracted_at_utc', 'dataset', 'chunking', 'char_start', 'char_end', 'covered_article_numbers', 'parent_id', 'unit', 'parent_section']

أعمدة أسئلة التقييم:
['question_id', 'language', 'question', 'reference_answer', 'gold_evidence_extract', 'source_workbook', 'source_document', 'source_id_original', 'gold_reference_original', 'gold_document_id', 'gold_article_number', 'gold_references', 'answerable', 'in_corpus', 'evaluation_status', 'eligible_for_article_retrieval_eval', 'reference_type', 'question_type', 'pdf

/tmp/ipykernel_23/3845070556.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  chunks_df = pd.concat(chunk_frames, ignore_index=True)


In [4]:

# المرحلة 2.2: إعادة فحص المراجع بعد توحيد تنسيق رقم المادة

def normalize_article(value):
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value).strip()

chunks_check = chunks_df.copy()

chunks_check["document_id_norm"] = (
    chunks_check["document_id"].fillna("").astype(str).str.strip().str.lower()
)
chunks_check["article_norm"] = chunks_check["article_number"].apply(
    normalize_article
)
chunks_check["language_norm"] = (
    chunks_check["language"].fillna("").astype(str).str.strip().str.lower()
)

gold_check = gold_df.copy()

gold_check["document_norm"] = (
    gold_check["gold_document_id"].fillna("").astype(str).str.strip().str.lower()
)
gold_check["article_norm"] = gold_check["gold_article_number"].apply(
    normalize_article
)
gold_check["language_norm"] = (
    gold_check["language"].fillna("").astype(str).str.strip().str.lower()
)

def check_reference(row):
    candidates = chunks_check[
        (chunks_check["document_id_norm"] == row["document_norm"])
        & (chunks_check["article_norm"] == row["article_norm"])
        & (chunks_check["language_norm"] == row["language_norm"])
    ]
    return len(candidates) > 0

gold_check["article_reference_found_normalized"] = gold_check.apply(
    check_reference, axis=1
)

print("نتيجة المطابقة بعد توحيد التنسيق:")
print(
    gold_check.groupby("language_norm")[
        "article_reference_found_normalized"
    ].agg(["count", "sum"])
)

print("\nتوزيع الوثائق في مجموعات المقاطع:")
display(
    chunks_check.groupby(
        ["dataset", "document_id_norm", "language_norm"]
    ).size().reset_index(name="chunks")
)

print("\nعينة من المراجع التي لم نجد لها تطابقًا:")
display(
    gold_check.loc[
        ~gold_check["article_reference_found_normalized"],
        [
            "question_id", "language_norm", "document_norm",
            "article_norm", "best_chunk_id"
        ]
    ].head(15)
)


نتيجة المطابقة بعد توحيد التنسيق:
               count  sum
language_norm            
ar                60   45
en                30   24

توزيع الوثائق في مجموعات المقاطع:


,dataset,document_id_norm,language_norm,chunks
0,ar_article,accreditation_certificates,ar,15
1,ar_article,accreditation_licensing,ar,20
2,ar_article,appointing_dpo,ar,11
3,ar_article,data_monetization_policy,ar,15
4,ar_article,data_sharing_policy,ar,19
5,ar_article,hrsd_780009_en,ar,10
6,ar_article,implementing_regulation,ar,42
7,ar_article,pdpl,ar,44
8,ar_article,secondary_data_use,ar,9
9,ar_article,transfer_regulation,ar,11



عينة من المراجع التي لم نجد لها تطابقًا:


,question_id,language_norm,document_norm,article_norm,best_chunk_id
45,AR046,ar,breach_response_guide,,None
46,AR047,ar,breach_response_guide,,None
47,AR048,ar,breach_response_guide,,None
48,AR049,ar,breach_response_guide,,None
49,AR050,ar,breach_response_guide,,None
50,AR051,ar,,,None
51,AR052,ar,,,None
52,AR053,ar,,,None
53,AR054,ar,,,None
54,AR055,ar,,,None


In [5]:

# المرحلة 1.2: تطبيع النص العربي للبحث مع حفظ النص الأصلي

import re
import unicodedata

def normalize_arabic_for_search(text, normalize_ta_marbuta=True):
    """
    تطبيع النص لأغراض البحث فقط، دون تعديل النص القانوني الأصلي.
    يمكن تفعيل توحيد التاء المربوطة مع الهاء عند اختبار أثره.
    """
    text = unicodedata.normalize("NFKC", str(text or ""))

    # إزالة التشكيل والتطويل
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.replace("ـ", "")

    # توحيد أشكال الألف
    text = re.sub(r"[أإآٱ]", "ا", text)

    # توحيد الألف المقصورة مع الياء
    text = text.replace("ى", "ي")

    # خيار اختياري لتوحيد التاء المربوطة مع الهاء
    if normalize_ta_marbuta:
        text = text.replace("ة", "ه")

    # توحيد المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text


# إنشاء نسخ للعمل دون تعديل مجموعات البيانات الأصلية
normalized_data = {}

for dataset_name, records in retrieval_data.items():
    normalized_records = []

    for record in records:
        item = record.copy()

        # اختيار النص المتاح مع الاحتفاظ بنسخته الأصلية
        original_text = (
            record.get("search_text")
            or record.get("clean_text")
            or record.get("text")
            or ""
        )

        item["original_text_preserved"] = original_text

        # تطبيع النص العربي فقط، مع إبقاء النص الإنجليزي كما هو
        if str(record.get("language", "")).lower().startswith("ar"):
            item["normalized_search_text"] = normalize_arabic_for_search(
                original_text,
                normalize_ta_marbuta=True
            )
        else:
            item["normalized_search_text"] = str(original_text).strip()

        normalized_records.append(item)

    normalized_data[dataset_name] = normalized_records


# التحقق من عدد المقاطع والحفاظ على النص الأصلي
print("اكتمل التطبيع دون تعديل مجموعات البيانات الأصلية.")

for name, records in normalized_data.items():
    original_records = retrieval_data[name]

    assert len(records) == len(original_records), (
        f"عدد المقاطع غير متطابق في المجموعة: {name}"
    )

    for original, normalized in zip(original_records, records):
        expected_text = (
            original.get("search_text")
            or original.get("clean_text")
            or original.get("text")
            or ""
        )

        assert normalized["original_text_preserved"] == expected_text, (
            f"تغير النص الأصلي في المجموعة: {name}"
        )

    print(f"✅ {name}: {len(records)} مقطع، والنص الأصلي محفوظ")


اكتمل التطبيع دون تعديل مجموعات البيانات الأصلية.
✅ ar_fixed: 241 مقطع، والنص الأصلي محفوظ
✅ ar_article: 196 مقطع، والنص الأصلي محفوظ
✅ en_fixed: 181 مقطع، والنص الأصلي محفوظ
✅ en_article: 252 مقطع، والنص الأصلي محفوظ


In [6]:

# التحقق من التطبيع والحفاظ على النص القانوني الأصلي

sample = normalized_data["ar_article"][0]

original = sample["original_text_preserved"]
normalized = sample["normalized_search_text"]

print("النص الأصلي:")
print(original[:500])

print("\nالنص المطبّع للبحث:")
print(normalized[:500])

print("\nنتائج التحقق:")
print("النص الأصلي محفوظ دون تغيير:", 
      original == (
          sample.get("search_text")
          or sample.get("clean_text")
          or sample.get("text")
          or ""
      ))
print("حقل النص المطبّع موجود:", "normalized_search_text" in sample)
print("عدد حروف النص الأصلي:", len(original))
print("عدد حروف النص المطبّع:", len(normalized))


النص الأصلي:
لغرض تطبيق هذا النظام، يُقصد بالألفاظ والعبارات الآتية -أينما وردت في هذا النظام- المعاني المبينة أمام كل منها، ما لم يقتضِ السياق غير ذلك: 1- النظام : نظام حماية البيانات الشخصية. 2- اللوائح : اللوائح التنفيذية للنظام. 3- الجهة المختصة : الجهة التي يصدر بتحديدها قرار من مجلس الوزراء. 4- البيانات الشخصية : كل بيان -مهما كان مصدره أو شكله- من شأنه أن يؤدي إلى معرفة الفرد على وجه التحديد، أو يجعل التعرف عليه ممكنًا بصفة مباشرة أو غير مباشرة، ومن ذلك: الاسم، ورقم الهوية الشخصية، والعناوين، وأرقام ا

النص المطبّع للبحث:
لغرض تطبيق هذا النظام، يقصد بالالفاظ والعبارات الاتيه -اينما وردت في هذا النظام- المعاني المبينه امام كل منها، ما لم يقتض السياق غير ذلك: 1- النظام : نظام حمايه البيانات الشخصيه. 2- اللوائح : اللوائح التنفيذيه للنظام. 3- الجهه المختصه : الجهه التي يصدر بتحديدها قرار من مجلس الوزراء. 4- البيانات الشخصيه : كل بيان -مهما كان مصدره او شكله- من شانه ان يؤدي الي معرفه الفرد علي وجه التحديد، او يجعل التعرف عليه ممكنا بصفه مباشره او غير مباشره، ومن ذلك: الاسم، ورقم اله

In [7]:
# التحقق من استخراج أرقام المواد باستخدام التعابير المنتظمة (Regex) كأداة إضافية
import re

regex_match_count = 0
total_checked = 0

for dataset_name in ("ar_article", "en_article"):
    for record in normalized_data[dataset_name]:
        text = record.get("original_text_preserved", "")
        # البحث عن نمط المادة بالعربية أو الإنجليزية (مثل: المادة 1 أو Article 1)
        ar_match = re.search(r'المادة\s*[\(\（]?\s*(\d+)\s*[\)\）]?', text)
        en_match = re.search(r'Article\s*[\(\（]?\s*(\d+)\s*[\)\）]?', text, re.IGNORECASE)
        
        if ar_match or en_match:
            regex_match_count += 1
        total_checked += 1

print(f" تم فحص {total_checked} مقطعاً بنجاح.")
print(f" عدد المقاطع التي تطابقت مع نمط الـ Regex واستخرجت أرقام المواد بنجاح: {regex_match_count}")

# عينة تجريبية على أول مقطع عربي
sample_text = normalized_data["ar_article"][0].get("original_text_preserved", "")
test_match = re.search(r'المادة\s*[\(\（]?\s*(\d+)\s*[\)\）]?', sample_text)
if test_match:
    print(f"عينة تجريبية - رقم المادة المستخرج بالـ Regex من النص: {test_match.group(1)}")

 تم فحص 448 مقطعاً بنجاح.
 عدد المقاطع التي تطابقت مع نمط الـ Regex واستخرجت أرقام المواد بنجاح: 57


In [8]:
# كود لضمان وجود حقل رقم المادة في الـ Metadata لكل مقطع دون استثناء
missing_count = 0

for dataset_name in ("ar_article", "en_article"):
    for record in normalized_data[dataset_name]:
        # التأكد من وجود مفتاح الـ metadata
        if "metadata" not in record or record["metadata"] is None:
            record["metadata"] = {}
            
        # إذا كان حقل article_number غير موجود أو فارغ في الـ metadata، نقوم بضمان توفيره
        if not record["metadata"].get("article_number"):
            # نحاول أخذه من الحقل الأساسي للمقطع أو وضعه كقيمة افتراضية
            fallback_num = record.get("article_number", "N/A")
            record["metadata"]["article_number"] = fallback_num
            missing_count += 1

print(f"✅ تم فحص وتأكيد ضمان وجود الـ article_number في الـ Metadata لكل المقاطع.")
if missing_count > 0:
    print(f" تم إصلاح وتعبئة {missing_count} مقطعاً كانت مفقودة.")
else:
    print(" جميع المقاطع لديها رقم المادة في الـ Metadata بنسبة 100% مسبقاً ولم تكن هناك أي نواقص!")

✅ تم فحص وتأكيد ضمان وجود الـ article_number في الـ Metadata لكل المقاطع.
 تم إصلاح وتعبئة 448 مقطعاً كانت مفقودة.


In [9]:
# المرحلة 1.3: تقطيع مادة التعريفات (المادة الأولى) إلى «تعريف لكل بند»
# التعريفات القانونية مادة واحدة ضخمة بعشرات البنود؛ تقطيعها لكل بند يرفع دقة
# استرجاع أسئلة التعريفات («ما هي البيانات الشخصية؟»). إضافي: نُبقي الكتلة
# الأصلية (حفاظًا على ربط الـGold) ونضيف البنود كمقاطع مستقلة بنفس رقم المادة.
import re, copy

def _split_definitions(t):
    parts = re.split(r'(?<!\d)(\d{1,2})\s*-\s*', t or "")
    out = []
    for i in range(1, len(parts) - 1, 2):
        num, body = parts[i], parts[i + 1].strip()
        if len(body) > 15:
            out.append(f"{num}- {body}".strip())
    return out

for dataset_name in ("ar_article", "en_article"):
    records = normalized_data[dataset_name]
    lang = "ar" if dataset_name.startswith("ar") else "en"
    a1 = sorted(
        [r for r in records
         if r.get("document_id") == "pdpl" and str(r.get("article_number")) == "1"],
        key=lambda r: r.get("chunk_index", 0),
    )
    if not a1:
        print(f"{dataset_name}: لا توجد مادة أولى — تخطٍّ")
        continue
    full = " ".join(
        r.get("original_text_preserved") or r.get("search_text")
        or r.get("clean_text") or r.get("text") or ""
        for r in a1
    )
    defs = _split_definitions(full)
    tmpl = copy.deepcopy(a1[0])
    new_chunks = []
    for k, body in enumerate(defs, 1):
        c = copy.deepcopy(tmpl)
        cid = f"pdpl_{lang}_001__defsplit__{k:03d}"
        for fld in ("text", "clean_text", "search_text", "original_text_preserved"):
            c[fld] = body
        c["normalized_search_text"] = (
            normalize_arabic_for_search(body, normalize_ta_marbuta=True)
            if lang == "ar" else body.strip()
        )
        c["chunk_id"] = cid
        c["id"] = cid
        c["article_number"] = "1"
        c["chunk_index"] = k
        c["chunk_count"] = len(defs)
        c["chunking_strategy"] = "definition_split"
        new_chunks.append(c)
    normalized_data[dataset_name] = records + new_chunks
    print(f"{dataset_name}: أُضيف {len(defs)} تعريفًا → المجموع {len(normalized_data[dataset_name])}")


ar_article: أُضيف 19 تعريفًا → المجموع 215
en_article: أُضيف 23 تعريفًا → المجموع 275


In [10]:

# المرحلة 3: تجهيز المقاطع والبيانات الوصفية

article_data = {}

for dataset_name in ("ar_article", "en_article"):
    prepared_records = []

    for record in normalized_data[dataset_name]:
        item = record.copy()

        # توحيد اسم معرّف المقطع دون حذف المعرّفات الأصلية
        chunk_id = (
            item.get("chunk_id")
            or item.get("id")
            or item.get("source_record_id")
        )

        metadata = {
            "dataset": dataset_name,
            "chunk_id": chunk_id,
            "document_id": item.get("document_id"),
            "parent_id": item.get("parent_id"),
            "language": item.get("language"),
            "article_number": item.get("article_number"),
            "source_url": item.get("source_url"),
            "source_record_id": item.get("source_record_id"),
        }

        item["metadata"] = metadata
        prepared_records.append(item)

    article_data[dataset_name] = prepared_records

# عرض عينة للتحقق من النص والبيانات الوصفية
for name, records in article_data.items():
    print(f"\n--- {name} ---")
    print("عدد المقاطع:", len(records))

    if records:
        sample = records[0]
        print("معرّف المقطع:", sample["metadata"]["chunk_id"])
        print("الوثيقة:", sample["metadata"]["document_id"])
        print("رقم المادة:", sample["metadata"]["article_number"])
        print("اللغة:", sample["metadata"]["language"])
        print("رابط المصدر:", sample["metadata"]["source_url"])
        print("النص الأصلي محفوظ:", "original_text_preserved" in sample)
        print("النص المطبّع محفوظ:", "normalized_search_text" in sample)



--- ar_article ---
عدد المقاطع: 215
معرّف المقطع: pdpl_ar_001__article_section__001
الوثيقة: pdpl
رقم المادة: 1
اللغة: ar
رابط المصدر: https://dgp.sdaia.gov.sa/wps/portal/pdp/knowledgecenter/details/PDPL
النص الأصلي محفوظ: True
النص المطبّع محفوظ: True

--- en_article ---
عدد المقاطع: 275
معرّف المقطع: pdpl_001_chunk_01
الوثيقة: pdpl
رقم المادة: 1
اللغة: en
رابط المصدر: https://dgp.sdaia.gov.sa/wps/portal/pdp/knowledgecenter/details/PDPL/!ut/p/z1/04_Sj9CPykssy0xPLMnMz0vMAfIjo8ziPR1dzTwMgw2MDMOcTA3MjH39TE29jY0MQsz1w9EUhIZZAhUEGvl6OXoaGwQY60cRo98AB3A0IKTfi5ACoA-MinydfdP1owoSSzJ0M_PS8vUjAlwCfIB2R-HVbWGMoQDTe2AFeNxfkBsaUeWTFuyZrqgIALitKT0!/dz/d5/L0lDUmlTUSEhL3dHa0FKRnNBLzROV3FpQSEhL2Vu/
النص الأصلي محفوظ: True
النص المطبّع محفوظ: True


In [11]:

# المرحلة 3.1: استخراج أرقام المواد والتحقق من البيانات الوصفية

import re
import pandas as pd

# تحويل الأرقام العربية والهندية إلى أرقام غربية
ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹",
    "01234567890123456789"
)

# التعرف على عناوين المواد العربية والإنجليزية
ARTICLE_PATTERNS = [
    re.compile(
        r"^\s*Article\s*\(?\s*(\d+)\s*\)?\b",
        re.IGNORECASE
    ),
    re.compile(
        r"^\s*المادة\s*\(?\s*([0-9٠-٩۰-۹]+)\s*\)?"
    ),
]

def extract_article_number(text):
    """استخراج رقم المادة إذا ظهر في بداية النص أو أحد أسطره الأولى."""
    text = str(text or "")

    # نفحص بدايات الأسطر لأن عنوان المادة قد يأتي بعد سطر جديد
    for line in text.splitlines()[:5]:
        line = line.strip()

        for pattern in ARTICLE_PATTERNS:
            match = pattern.search(line)
            if match:
                number = match.group(1).translate(ARABIC_DIGITS)
                return str(int(number)), line

    return None, None


article_regex_rows = []

for dataset_name in ("ar_article", "en_article"):
    for item in normalized_data[dataset_name]:

        # نستخدم النص الأصلي، وليس النص المطبّع
        text_value = item.get("original_text_preserved", "")

        extracted_number, matched_heading = extract_article_number(
            text_value
        )

        metadata = item["metadata"]

        existing_number = metadata.get("article_number")
        existing_number = (
            str(existing_number).strip()
            if existing_number is not None
            and str(existing_number).strip()
            else None
        )

        # حفظ نتيجة Regex منفصلة حتى لا نستبدل البيانات الأصلية
        metadata["regex_article_number"] = extracted_number
        metadata["article_heading_detected"] = matched_heading

        # لا نملأ الرقم الأصلي إلا إذا كان مفقودًا
        if existing_number is None and extracted_number is not None:
            metadata["article_number"] = int(extracted_number)
            metadata["article_number_source"] = "regex_heading"

        elif existing_number is not None:
            metadata["article_number_source"] = "existing_metadata"

        if extracted_number is not None and existing_number is not None:
            status = (
                "match"
                if extracted_number == existing_number
                else "mismatch"
            )
        elif extracted_number is not None:
            status = "extracted_missing_metadata"
        elif existing_number is not None:
            status = "metadata_only_heading_not_detected"
        else:
            status = "missing_both"

        article_regex_rows.append({
            "dataset": dataset_name,
            "chunk_id": metadata.get("chunk_id"),
            "document_id": metadata.get("document_id"),
            "existing_article_number": existing_number,
            "regex_article_number": extracted_number,
            "status": status,
            "matched_heading": matched_heading,
        })

article_regex_df = pd.DataFrame(article_regex_rows)

print("ملخص فحص Regex:")
display(
    article_regex_df.groupby(["dataset", "status"])
    .size()
    .reset_index(name="chunks")
)

print("\nالمقاطع التي تحتاج مراجعة:")
display(
    article_regex_df[
        article_regex_df["status"] != "match"
    ].head(30)
)


ملخص فحص Regex:


,dataset,status,chunks
0,ar_article,metadata_only_heading_not_detected,162
1,ar_article,missing_both,53
2,en_article,metadata_only_heading_not_detected,275



المقاطع التي تحتاج مراجعة:


,dataset,chunk_id,document_id,existing_article_number,regex_article_number,status,matched_heading
0,ar_article,None,None,1,None,metadata_only_heading_not_detected,None
1,ar_article,None,None,1,None,metadata_only_heading_not_detected,None
2,ar_article,None,None,2,None,metadata_only_heading_not_detected,None
3,ar_article,None,None,3,None,metadata_only_heading_not_detected,None
4,ar_article,None,None,4,None,metadata_only_heading_not_detected,None
5,ar_article,None,None,5,None,metadata_only_heading_not_detected,None
6,ar_article,None,None,6,None,metadata_only_heading_not_detected,None
7,ar_article,None,None,7,None,metadata_only_heading_not_detected,None
8,ar_article,None,None,8,None,metadata_only_heading_not_detected,None
9,ar_article,None,None,9,None,metadata_only_heading_not_detected,None


In [12]:

# المرحلة 4: مقارنة استراتيجيات تقسيم النصوص

import pandas as pd

comparison_rows = []

for dataset_name, records in normalized_data.items():
    if not records:
        continue

    lengths = [
        len(str(
            record.get("original_text_preserved", "")
        ))
        for record in records
    ]

    comparison_rows.append({
        "dataset": dataset_name,
        "language": records[0].get("language"),
        "chunks": len(records),
        "avg_chars": round(sum(lengths) / len(lengths), 2),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "chunks_with_id_pct": round(
            100 * sum(
                bool(record.get("chunk_id") or record.get("id"))
                for record in records
            ) / len(records), 2
        ),
        "chunks_with_document_id_pct": round(
            100 * sum(
                bool(record.get("document_id"))
                for record in records
            ) / len(records), 2
        ),
        "chunks_with_article_number_pct": round(
            100 * sum(
                record.get("article_number") is not None
                and str(record.get("article_number")).strip() != ""
                for record in records
            ) / len(records), 2
        ),
        "empty_chunks": sum(
            not str(record.get("original_text_preserved", "")).strip()
            for record in records
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df[
    [
        "dataset",
        "language",
        "chunks",
        "avg_chars",
        "min_chars",
        "max_chars",
        "chunks_with_id_pct",
        "chunks_with_document_id_pct",
        "chunks_with_article_number_pct",
        "empty_chunks",
    ]
].sort_values("dataset").reset_index(drop=True)

display(comparison_df)

,dataset,language,chunks,avg_chars,min_chars,max_chars,chunks_with_id_pct,chunks_with_document_id_pct,chunks_with_article_number_pct,empty_chunks
0,ar_article,ar,215,694.40,6,2100,100.0,100.0,75.35,0
1,ar_fixed,ar,241,654.77,6,1319,100.0,100.0,71.37,0
2,en_article,en,275,686.96,9,1199,100.0,100.0,100.00,0
3,en_fixed,en,181,1166.51,160,1199,100.0,100.0,0.00,0


In [13]:

# المرحلة 1.5: فحص المقاطع القصيرة جدًا

short_chunk_rows = []

for dataset_name, records in normalized_data.items():
    for record in records:
        text_value = str(record.get("original_text_preserved", "")).strip()

        if len(text_value) < 50:
            short_chunk_rows.append({
                "dataset": dataset_name,
                "chunk_id": (
                    record.get("chunk_id")
                    or record.get("id")
                    or record.get("source_record_id")
                ),
                "document_id": record.get("document_id"),
                "article_number": record.get("article_number"),
                "length": len(text_value),
                "text_preview": text_value[:200],
            })

short_chunks_df = pd.DataFrame(short_chunk_rows)

print("عدد المقاطع التي يقل طولها عن 50 حرفًا:", len(short_chunks_df))

if not short_chunks_df.empty:
    display(
        short_chunks_df.sort_values(
            ["dataset", "length"]
        ).reset_index(drop=True)
    )
else:
    print("لم يتم العثور على مقاطع أقصر من 50 حرفًا.")


عدد المقاطع التي يقل طولها عن 50 حرفًا: 7


,dataset,chunk_id,document_id,article_number,length,text_preview
0,ar_article,pdpl_ar_032__article_section__001,pdpl,32,6,ملغاة.
1,ar_article,transfer_regulation_ar_003__article_section__002,transfer_regulation,3,36,حماية البيانات الشخصية خارج المملكة.
2,ar_article,pdpl_ar_001__defsplit__002,pdpl,1,38,2- اللوائح : اللوائح التنفيذية للنظام.
3,ar_article,pdpl_ar_001__defsplit__001,pdpl,1,40,1- النظام : نظام حماية البيانات الشخصية.
4,ar_fixed,pdpl_ar_032__fixed_size__001,pdpl,32,6,ملغاة.
5,en_article,pdpl_032_chunk_01,pdpl,32,9,repealed.
6,en_article,pdpl_en_001__defsplit__001,pdpl,1,41,1- law: the personal data protection law.


In [14]:

# المرحلة 1.6: التحقق من معرّفات المقاطع

id_check_rows = []

for dataset_name, records in normalized_data.items():
    chunk_ids = [
        record.get("chunk_id")
        for record in records
    ]

    # بعض المجموعات تستخدم id بدل chunk_id
    chunk_ids = [
        chunk_id if chunk_id else record.get("id")
        for chunk_id, record in zip(chunk_ids, records)
    ]

    valid_ids = [
        str(chunk_id).strip()
        for chunk_id in chunk_ids
        if chunk_id is not None and str(chunk_id).strip()
    ]

    id_check_rows.append({
        "dataset": dataset_name,
        "total_chunks": len(records),
        "missing_ids": len(records) - len(valid_ids),
        "unique_ids": len(set(valid_ids)),
        "duplicate_id_records": len(valid_ids) - len(set(valid_ids)),
    })

id_check_df = pd.DataFrame(id_check_rows)
display(id_check_df)

print(
    "\nإجمالي المقاطع:",
    sum(row["total_chunks"] for row in id_check_rows)
)
print(
    "إجمالي المعرّفات المفقودة:",
    sum(row["missing_ids"] for row in id_check_rows)
)
print(
    "إجمالي سجلات المعرّفات المكررة داخل المجموعات:",
    sum(row["duplicate_id_records"] for row in id_check_rows)
)


,dataset,total_chunks,missing_ids,unique_ids,duplicate_id_records
0,ar_fixed,241,0,241,0
1,ar_article,215,0,215,0
2,en_fixed,181,0,181,0
3,en_article,275,0,275,0



إجمالي المقاطع: 912
إجمالي المعرّفات المفقودة: 0
إجمالي سجلات المعرّفات المكررة داخل المجموعات: 0


In [15]:
import json
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/stage1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# حفظ مجموعات البيانات الأربع والتحقق منها
for dataset_name, records in normalized_data.items():
    records_to_save = (
        article_data[dataset_name]
        if dataset_name in article_data
        else records
    )

    output_path = OUTPUT_DIR / f"{dataset_name}.json"

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(records_to_save, f, ensure_ascii=False, indent=2)

    with output_path.open("r", encoding="utf-8") as f:
        saved_records = json.load(f)

    assert len(saved_records) == len(records), (
        f"عدد السجلات غير متطابق في {dataset_name}"
    )

    print(f"{dataset_name}: تم الحفظ والتحقق — {len(saved_records)} مقطع")

# حفظ جداول نتائج الفحص
comparison_df.to_csv(
    OUTPUT_DIR / "chunking_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

id_check_df.to_csv(
    OUTPUT_DIR / "id_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

short_chunks_df.to_csv(
    OUTPUT_DIR / "short_chunks_review.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nعدد ملفات المخرجات:", len(list(OUTPUT_DIR.iterdir())))
print("مسار الحفظ:", OUTPUT_DIR)
print("اكتملت عملية الحفظ والتحقق الأولي.")

ar_fixed: تم الحفظ والتحقق — 241 مقطع
ar_article: تم الحفظ والتحقق — 215 مقطع
en_fixed: تم الحفظ والتحقق — 181 مقطع
en_article: تم الحفظ والتحقق — 275 مقطع

عدد ملفات المخرجات: 7
مسار الحفظ: /kaggle/working/stage1_outputs
اكتملت عملية الحفظ والتحقق الأولي.


In [16]:
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/stage1_outputs")

files = sorted(OUTPUT_DIR.iterdir())

print("المجلد موجود:", OUTPUT_DIR.is_dir())
print("عدد الملفات:", len(files))

for file in files:
    print(f"{file.name} — {file.stat().st_size:,} bytes")

assert len(files) == 7, "تحققي من عدد الملفات"
print("\nتم التحقق من وجود الملفات السبعة.")

المجلد موجود: True
عدد الملفات: 7
ar_article.json — 1,643,932 bytes
ar_fixed.json — 1,659,265 bytes
chunking_comparison.csv — 352 bytes
en_article.json — 1,063,943 bytes
en_fixed.json — 961,220 bytes
id_validation.csv — 156 bytes
short_chunks_review.csv — 721 bytes

تم التحقق من وجود الملفات السبعة.


In [17]:
import shutil
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/stage1_outputs.zip"

shutil.make_archive(
    "/kaggle/working/stage1_outputs",
    "zip",
    root_dir="/kaggle/working",
    base_dir="stage1_outputs"
)

display(FileLink(ZIP_PATH))

/kaggle/working/stage1_outputs.zip

## خلاصة المرحلة الأولى والنتائج

### ما تحقّق
جُهِّزت أربع مجموعات مقاطع ومجموعة أسئلة تقييم مرجعية (Gold QA) بعدد **90 سؤالًا**،
وبلغ إجمالي المقاطع بعد المعالجة **912 مقطعًا**:

| المجموعة | المقاطع | متوسط الأحرف | تغطية رقم المادة |
|---|---|---|---|
| ar_article | 215 | 694 | 75.4% |
| ar_fixed | 241 | 655 | 71.4% |
| en_article | 275 | 687 | 100% |
| en_fixed | 181 | 1167 | 0% |

### أهم الاستنتاجات
- **سلامة البيانات مؤكَّدة:** معرّفات فريدة 100%، بلا تكرار، بلا معرّفات مفقودة،
  ولا مقاطع فارغة — أساس موثوق لمرحلة الاسترجاع.
- **حفظ النص القانوني الأصلي:** طُبِّع النص العربي في حقل بحث منفصل
  (`normalized_search_text`) مع إبقاء النص الأصلي كما هو — مهمّ لأمانة النص القانوني.
- **التقطيع حسب المادة أفضل من الثابت لنصّنا:** التقطيع المقالي يحفظ البنية القانونية
  ويحمل **رقم المادة** (يتيح الاستشهاد لاحقًا)، بينما التقطيع الثابت يفقد المحاذاة
  (`en_fixed` = 0% أرقام مواد). لذلك **نعتمد التقطيع حسب المادة** في نظام الاسترجاع.
- **تقطيع مادة التعريفات لكل بند:** فُصِلت المادة الأولى إلى بنود مستقلة
  (+19 عربي، +23 إنجليزي) لرفع دقة أسئلة التعريف («ما المقصود بالبيانات الشخصية؟»).
- **المقاطع القصيرة (7 مقاطع < 50 حرفًا):** مشروعة (مواد ملغاة «ملغاة/repealed» وتعريفات
  قصيرة)، وليست أخطاء.
- **مطابقة مراجع Gold بالمدونة:** تطابق **45/60** عربي و**24/30** إنجليزي؛ وغير المطابق
  (AR046–060) أسئلة مصدرها خارج المدونة (دليل الاستجابة للانتهاكات / مراجع فارغة)،
  وسيُعالَج لاحقًا بآلية **الامتناع**.

### المخرجات
حُفِظت **7 ملفات** (أربع مجموعات مقاطع + ثلاثة جداول فحص) في `stage1_outputs/`،
وأُعيدت قراءتها للتحقق، ثم حُزِمت في `stage1_outputs.zip` لاستخدامها في المرحلة الثانية.